### INGEST

In [0]:
import logging

logging.basicConfig(level=logging.INFO)

logger = logging.getLogger(__name__)

In [0]:
dbutils.fs.ls('/Volumes/workspace/default/my_files')

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
logger.info("Starting medical appointment pipeline")

In [0]:
df = spark.read.format('csv').option('header', 'true').load('/Volumes/workspace/default/my_files/KaggleV2-May-2016.csv')
logger.info("DataFrame loaded successfully")
logger.info("Total records: %s", df.count())

In [0]:
my_struct_schema = StructType([
  StructField('PatientId', LongType(), True),
  StructField('AppointmentID', LongType(), True),
  StructField('Gender', StringType(), True),
  StructField('ScheduledDay', TimestampType(), True),
  StructField('AppointmentDay', TimestampType(), True),
  StructField('Age', IntegerType(), True),
  StructField('Neighbourhood', StringType(), True),
  StructField('Scholarship', IntegerType(), True),
  StructField('Hipertension', IntegerType(), True),
  StructField('Diabetes', IntegerType(), True),
  StructField('Alcoholism', IntegerType(), True),
  StructField('Handcap', IntegerType(), True),
  StructField('SMS_received', IntegerType(), True),
  StructField('No-show', StringType(), True)
])

In [0]:
df = spark.read.format('csv').schema(my_struct_schema).option('header', 'true').load('/Volumes/workspace/default/my_files/KaggleV2-May-2016.csv')
logger.info("DataFrame loaded successfully")
df.display()

In [0]:
output_path = "/Volumes/workspace/default/my_files/medical_appointment/ingest"

df.write.mode("overwrite").parquet(output_path)

In [0]:
logger.info("Data ingested")

### Clean and transform

In [0]:
df = df.withColumnRenamed('PatientId', 'Patient_ID')\
    .withColumnRenamed('AppointmentID', 'Appointment_ID')\
    .withColumnRenamed('Gender', 'Gender')\
    .withColumnRenamed('ScheduledDay', 'Scheduled_Day')\
    .withColumnRenamed('AppointmentDay', 'Appointment_Day')\
    .withColumnRenamed('Age', 'Age')\
    .withColumnRenamed('Neighbourhood', 'Neighbourhood')\
    .withColumnRenamed('No-show', 'NoShow')\
    .withColumnRenamed('Hipertension', 'Hypertension')\
    .withColumnRenamed('SMS_received', 'SMS_Received')\
    .withColumnRenamed('Scholarship', 'Scholarship')\
    .withColumnRenamed('Diabetes', 'Diabetes')\
    .withColumnRenamed('Alcoholism', 'Alcoholism')\
    .withColumnRenamed('Handcap', 'Handicap')
logger.info("Columns renamed")
df.display()

In [0]:
df.filter(col('Patient_ID').isNull()).count()

In [0]:
df = df.withColumn('Scheduled_Day', col('Scheduled_Day').cast('timestamp'))\
    .withColumn('Appointment_Day', col('Appointment_Day').cast('timestamp'))\
    .withColumn('Patient_ID', col('Patient_ID').cast('double'))
logger.info("Scheduled_Day and Appointment_Day columns cast to timestamp")
df.display()

In [0]:
df = df.withColumn('invalid_age', \
    when((col("Age") < 0)| (col("Age") > 100), True).otherwise(False))\
    .withColumn('invalid_date_diff', when(to_date("Scheduled_Day") > to_date("Appointment_Day"), True).otherwise(False))
logger.info("Invalid age and date diff columns created")
df.display()

In [0]:
df.groupBy('Appointment_ID').count().filter(col('count') > 1).display()
df.filter(col('Appointment_ID').isNull()).display()

In [0]:
df = df.withColumn('NoShow', col('NoShow').cast('boolean'))
logger.info("NoShow column cast to boolean")
df.display()

In [0]:
df = df.withColumn('Missed_Appointment', col('NoShow') == 'Yes').drop('NoShow')
logger.info("NoShow column renamed to Missed_Appointment")
df.display()

In [0]:
df = df.withColumn('wait_days', datediff('Appointment_Day', 'Scheduled_Day'))
logger.info("Wait days column created")
df.display()

In [0]:
bad = df.filter(col('invalid_age') | col('invalid_date_diff')).count()
print(f"Dropping {bad} invalid rows")  
df = df.filter(~col('invalid_age') & ~col('invalid_date_diff'))

In [0]:
df = df.withColumn("age_group", \
    when((col("age")).isNull(), "Unknown")\
    .when((col("age") ) < 0, "Invalid_age")\
    .when((col("age")) <18, "Under 18")\
    .when((col("age")) <36, "18-35")\
    .when((col("age")) <51, "36-50")\
    .when((col("age")) <66, "51-65")\
    .otherwise("Above 65"))
logger.info("Age group column created")
df.display() 

In [0]:
df = df.withColumn('appointment_weekday', (date_format(col('Appointment_Day'), 'EEEE')))
df.display()
logger.info("Appointment weekday column created")

In [0]:
output_path = "/Volumes/workspace/default/my_files/medical_appointment/transform"

df.write.mode("overwrite").partitionBy('Appointment_Day').parquet(output_path)

In [0]:
logger.info("Silver layer created successfully")

### AGGREGATE

#### SUMMARY

In [0]:
def calculate_no_show_rate(df, group_column):
    return (
        df.groupBy(group_column)
        .agg(
            count('*').alias('total_appointments'),
            avg(
                col('Missed_Appointment').cast('int')
            ).alias('no_show_rate')
        )
        .withColumn(
            'no_show_rate',
            round(col('no_show_rate') * 100,2)
        )
    )

In [0]:
gold_no_show_by_age = calculate_no_show_rate(
    df, 'age_group'
)
gold_no_show_by_age.display()
logger.info("Gold summary created: no-show rate by age group")

In [0]:
gold_no_show_by_neighbourhood = calculate_no_show_rate(
    df, 'Neighbourhood'
)
gold_no_show_by_neighbourhood.display()
logger.info("Gold summary created: no-show rate by neighbourhood")

In [0]:
gold_no_show_by_wait_time = calculate_no_show_rate(
    df, 'wait_days'
).orderBy(col('wait_days').asc())
gold_no_show_by_wait_time.display()
logger.info("Gold summary created: no-show rate by wait time column")

In [0]:
gold_no_show_by_sms = calculate_no_show_rate(
    df, 'SMS_received'
)
gold_no_show_by_sms.display()
logger.info("Gold summary created: no-show rate by sms")

In [0]:
gold_no_show_by_scholarship = calculate_no_show_rate(
    df, 'Scholarship'
)
gold_no_show_by_scholarship.display()
logger.info("Gold summary created: no-show rate by scholarship")

### LOAD

In [0]:
gold_parquet_config = {
    "gold_no_show_by_age": (
        gold_no_show_by_age,
        "age_group"
    ),
    "gold_no_show_by_neighbourhood": (
        gold_no_show_by_neighbourhood,
        "Neighbourhood"
    ),
    "gold_no_show_by_wait_time": (
        gold_no_show_by_wait_time,
        "wait_days"
    ),
    "gold_no_show_by_sms": (
        gold_no_show_by_sms,
        "SMS_received"
    ),
    "gold_no_show_by_scholarship": (
        gold_no_show_by_scholarship,
        "Scholarship"
    )
}

base_path = (
    "/Volumes/workspace/default/my_files/"
    "medical_appointment/gold"
)

for table_name, (table_df, partition_column) in gold_parquet_config.items():
    (
        table_df.write
        .format("parquet")
        .mode("overwrite")
        .save(f"{base_path}/{table_name}")
    )

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/default/my_files/medical_appointment/"
    )
)